# Neural Preset · 阶段 0（Kaggle T4）

对应 `REPRODUCTION_PLAN.md` §10。目标只有一个：**流程跑通**，不追求论文效果。

## 跑之前

1. Notebook 右侧 `Settings → Accelerator: GPU T4 x2`（代码只用 `cuda:0`，第二张卡不会用到）；
2. `Settings → Internet`：**开**最省事（torchvision 会下 21 MB 的 EfficientNet-B0 权重）。
   关网也能跑：把 `efficientnet_b0-*.pth` 放进数据集的 `weights/` 目录，`pretrained.mode: local` + 填 `ckpt`；
3. 需要 3 个 Data 挂载（也可以合并成一个）：

| 挂载名 | 内容 |
|---|---|
| `neuralpreset-code` | 本仓库打成的 `code.zip`（`git archive` 或直接 zip，**不含 data/**） |
| `coco10k` | `images/`（10k 张 COCO）+ `coco10k.txt`（清单）+ `val/content/`（200 张留出） |
| `cubes` | 308 个 `.cube`（默认后端直接读，无需预先打包）；可选 `lut33_u8.npy` 走 npy 后端 |

风格图目录（评测用）建议也放进 `coco10k`：`val/style/`（200 张色调明显的图 + 29 张 val29）。

**依赖**：默认 LUT 后端要 `pillow-lut`（纯 Python wheel，约 40 KB）。有网就 `pip install pillow-lut`；
没网就把它和清单一起放进 `neuralpreset-code`，cell 2 改成装本地 wheel。

**首次运行几乎一定要改一两处**（路径 / dtype / shape）。别改 notebook 里的逻辑，改包里的代码，
本地提交后再重新上传 zip —— 这样第二次就不用记忆「上次在哪个 cell 里手改过」。

In [ ]:
# 1) 解压代码 + 环境自检
import os, sys, shutil, subprocess

IN = "/kaggle/input"
WORK = "/kaggle/working"
CODE_DIR = f"{WORK}/code"

SRC_ZIP = f"{IN}/neuralpreset-code/code.zip"
if os.path.isdir(CODE_DIR):
    shutil.rmtree(CODE_DIR)
os.makedirs(CODE_DIR, exist_ok=True)
subprocess.run(["unzip", "-q", "-o", SRC_ZIP, "-d", CODE_DIR], check=True)
sys.path.insert(0, CODE_DIR)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) 装 pillow-lut（默认 LUT 后端）+ 数一下 cube 文件
!pip install -q pillow-lut

import glob
CUBES = f"{IN}/cubes"          # 这个目录里放 308 个 .cube（可以带子目录）
cubes = sorted(glob.glob(f"{CUBES}/**/*.cube", recursive=True))
print(f"找到 {len(cubes)} 个 .cube，例如：{cubes[:3]}")
assert cubes, f"{CUBES} 下没有 .cube —— 检查挂载目录名"

# 可选后端（LUT 涨到几千条时才需要）
# !python {CODE_DIR}/scripts/prepare_lut.py --cube-dir {CUBES} --out-npy {WORK}/lut33_u8.npy

In [ ]:
# 3) 训练：先冒烟（约 1 分钟、能暴露 90% 的路径/形状错误），再全量
CFG = f"{CODE_DIR}/configs/phase0.yaml"

!python -m neuralpreset.train --config {CFG} --smoke --out {WORK}/out_smoke --limit 300

In [ ]:
# 4) 全量 10k × 8 epoch。中断后回来重跑这一格（--resume auto 会接上 out/ 里最新的 ckpt）
!python -m neuralpreset.train --config {CFG} --out {WORK}/out

In [ ]:
# 5) 出图 + 指标（A4/A5）
CKPT = f"{WORK}/out/last.pt"
VAL_C = f"{IN}/coco10k/val/content"
VAL_S = f"{IN}/coco10k/val/style"

!python -m neuralpreset.infer --ckpt {CKPT} \
    --content {VAL_C}/0001.jpg --style {VAL_S}/0001.jpg \
    --out {WORK}/demo.png --matrix --compare

!python -m neuralpreset.evaluate --ckpt {CKPT} \
    --content-dir {VAL_C} --style-dir {VAL_S} \
    --n-pairs 2500 --out {WORK}/eval --patch 512

In [ ]:
# 6) 归档（A7）：/kaggle/working 里的文件才是能下载的产物
import glob, json, os, shutil

os.makedirs(f"{WORK}/out", exist_ok=True)
for p in glob.glob(f"{WORK}/demo*.png") + glob.glob(f"{WORK}/eval/metrics.json"):
    shutil.copy2(p, f"{WORK}/out/")

archive = shutil.make_archive(f"{WORK}/neuralpreset_stage0", "zip", WORK, "out")
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
print(open(f"{WORK}/out/metrics.jsonl").read().strip().split("\n")[-1])